# XGBoost vs. HistGradientBoosting on the hand-raise data (20 / 20 / 20 classes)

**Question.** Would XGBoost, alone or with the TCN, predict my hand raises better than the trees in
`hand_raise_pipeline.ipynb` (scikit-learn's HistGradientBoosting) or the final model (a logistic regression on the
MLP and TCN scores)?

**Design.** Only **train** classes are used; validation and test are not touched, so the main notebook's
"test scored once" rule still holds. From the train classes, 60 are drawn at random and split into
**20 to fit, 20 to tune** (tree size, number of trees, C) **and 20 to compare**. Twenty classes is small and AP on
rare events is noisy, so this is repeated for **5 random draws**, and each comparison comes with a 95% interval
from resampling the 20 comparison classes.

| Model | Inputs |
|---|---|
| **HGB** | The 68 structured inputs; the main notebook's settings (learning rate 0.05, L2 1, 80% of inputs per split) and its 3 tree sizes |
| **XGBoost** | Same inputs; matched settings (learning rate 0.05, `reg_lambda` 1, 80% of inputs per split, leaf-wise growth with 15 / 31 / 63 leaves), each with and without up-weighting the rare raises (`scale_pos_weight`) |
| **HGB + MLP + TCN** | The 68 inputs + the saved MLP and TCN scores |
| **XGBoost + MLP + TCN** | The 68 inputs + the saved MLP and TCN scores |
| **MLP + TCN, logistic** | Just the two scores, as in the final model |

The MLP and TCN scores are the saved out-of-fold train scores from the main notebook (each class scored by a fold
model that never saw it). The 7 cosine similarities the main notebook also gave the trees need the 4.4 GB text
vectors, which are not in the shared data, so they are left out here.

**Metric.** Average precision (AP) on the comparison classes, as in the main notebook. AP for always guessing
"raise" equals the share of moments that are raises, so it is printed alongside.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
DRIVE_FILE_ID = "1IlGYH6Flux__kAginY3hwFz6wuLQmNbS"   # pipeline1_data_private_v3.zip, as in hand_raise_pipeline.ipynb
if IN_COLAB:
    import subprocess
    import zipfile

    repo = Path("/content/cs156-ml")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Brady-Rutherford/cs156-ml.git",
                        str(repo)], check=True)
    os.chdir(repo / "pipeline1")
    sys.path.insert(0, str(repo / "pipeline1"))
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    auth.authenticate_user()                         # sign in with an account the data file is shared with
    zp = Path("/content/pipeline1_data.zip")
    if not Path("data/tables").exists():
        request = build("drive", "v3").files().get_media(fileId=DRIVE_FILE_ID, supportsAllDrives=True)
        with open(zp, "wb") as fh:
            down, done = MediaIoBaseDownload(fh, request, chunksize=64 << 20), False
            while not done:
                _, done = down.next_chunk()
        zipfile.ZipFile(zp).extractall(".")
else:
    sys.path.insert(0, str(Path.cwd()))

from oncue import config, flex

df = flex.load()                                     # counting moments with y, split and the trees' inputs
cols = flex.gbt_inputs(df.columns)
have_scores = (flex.out_dir() / "mlp.pkl").exists() and (flex.out_dir() / "tcn.pkl").exists()
if have_scores:
    S = flex.specialist_scores(df, "mlp", "tcn")     # logits; out-of-fold on train rows
else:
    print("saved MLP / TCN results not found: only the HGB vs XGBoost comparison will run")
train_classes = np.sort(df.loc[df["split"] == "train", "class_id"].unique())
print(f"{len(df):,} moments, {len(cols)} inputs; {len(train_classes)} train classes; "
      f"share of moments that are raises (train): {df.loc[df['split'] == 'train', 'y'].mean():.4f}")

## The models

In [ ]:
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

STEP, MAX_TREES = flex.STAGE_EVERY, flex.GBT_MAX_TREES   # 20, 600 as in the main notebook

# min_child_weight is a sum of p(1 - p) over a leaf's rows; with raises at a few percent each row adds about
# 0.02-0.03, so 5 and 10 are roughly HGB's 200 and 400 rows per leaf.
XGB_BASE = dict(learning_rate=0.05, reg_lambda=1.0, colsample_bynode=0.8, tree_method="hist",
                grow_policy="lossguide", max_depth=0, enable_categorical=True, n_estimators=MAX_TREES,
                random_state=config.SEED, n_jobs=-1)
XGB_GRID = [{"max_leaves": 15, "min_child_weight": 5}, {"max_leaves": 31, "min_child_weight": 5},
            {"max_leaves": 63, "min_child_weight": 10}]


def pick_hgb(Xf, yf, Xt, yt):
    """Each of the main notebook's 3 tree sizes with up to 600 trees; tuning-class AP every 20 trees picks
    (size, trees). Returns the refitted model and its setting."""
    best = None
    for st in flex.GBT_GRID:
        m = flex._hgb(st, MAX_TREES).fit(Xf, yf)
        for n, p in enumerate(m.staged_predict_proba(Xt), 1):
            if n % STEP == 0:
                ap = average_precision_score(yt, p[:, 1])
                if best is None or ap > best[0]:
                    best = (ap, st, n)
    return flex._hgb(best[1], best[2]).fit(Xf, yf), f"{best[1]['max_leaf_nodes']} leaves, {best[2]} trees"


def pick_xgb(Xf, yf, Xt, yt):
    """3 tree sizes x (no up-weighting, sqrt(negatives / positives)) with up to 600 trees; tuning AP every 20 trees."""
    spw_up = float(np.sqrt((len(yf) - yf.sum()) / max(yf.sum(), 1)))
    best = None
    for st in XGB_GRID:
        for spw in (1.0, spw_up):
            m = xgb.XGBClassifier(**XGB_BASE, **st, scale_pos_weight=spw).fit(Xf, yf)
            for n in range(STEP, MAX_TREES + 1, STEP):
                ap = average_precision_score(yt, m.predict_proba(Xt, iteration_range=(0, n))[:, 1])
                if best is None or ap > best[0]:
                    best = (ap, m, n, st, spw)
    _, m, n, st, spw = best

    class Truncated:                                  # predict with the chosen number of trees
        def predict_proba(self, X):
            return m.predict_proba(X, iteration_range=(0, n))
    return Truncated(), f"{st['max_leaves']} leaves, {n} trees, pos. weight {spw:.1f}"


def pick_logistic(Xf, yf, Xt, yt):
    best = None
    for C in flex.LR_C_GRID:
        m = LogisticRegression(C=C, max_iter=2000).fit(Xf, yf)
        ap = average_precision_score(yt, m.predict_proba(Xt)[:, 1])
        if best is None or ap > best[0]:
            best = (ap, m, C)
    return best[1], f"C = {best[2]}"


def class_bootstrap_diff(cls, y, p_new, p_old, B=1000, seed=0):
    """95% interval for AP(new) - AP(old), resampling the comparison classes with replacement."""
    rng = np.random.default_rng(seed)
    idx = pd.Series(np.arange(len(y))).groupby(np.asarray(cls)).indices
    groups = list(idx.values())
    out = []
    for _ in range(B):
        rows = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        if y[rows].sum() == 0:
            continue
        out.append(average_precision_score(y[rows], p_new[rows]) - average_precision_score(y[rows], p_old[rows]))
    return np.percentile(out, [2.5, 97.5])

## Five draws of 20 / 20 / 20 train classes

In [ ]:
N_DRAWS, PER = 5, 20
rng = np.random.default_rng(config.SEED)
pos_by_class = df[df["split"] == "train"].groupby("class_id")["y"].sum()
X_all = df[cols]
if have_scores:
    X_plus = pd.concat([df[cols], S], axis=1)

rows, t0 = [], time.time()
for d in range(N_DRAWS):
    while True:                                       # every part needs at least a few raises
        c = rng.permutation(train_classes)[:3 * PER]
        parts = {"fit": c[:PER], "tune": c[PER:2 * PER], "compare": c[2 * PER:]}
        if all(pos_by_class.loc[v].sum() >= 5 for v in parts.values()):
            break
    m = {k: df["class_id"].isin(v).to_numpy() for k, v in parts.items()}
    y = {k: df["y"].to_numpy()[m[k]] for k in m}
    cls_cmp = df["class_id"].to_numpy()[m["compare"]]
    models = {"HGB": (pick_hgb, X_all), "XGBoost": (pick_xgb, X_all)}
    if have_scores:
        models |= {"HGB + MLP + TCN": (pick_hgb, X_plus), "XGBoost + MLP + TCN": (pick_xgb, X_plus),
                   "MLP + TCN, logistic": (pick_logistic, S)}
    preds = {}
    for name, (pick, X) in models.items():
        model, chosen = pick(X[m["fit"]], y["fit"], X[m["tune"]], y["tune"])
        preds[name] = model.predict_proba(X[m["compare"]])[:, 1]
        rows.append({"draw": d, "model": name, "AP": average_precision_score(y["compare"], preds[name]),
                     "chosen": chosen, "compare raises": int(y["compare"].sum()),
                     "chance AP": y["compare"].mean()})
    for new, old in (("XGBoost", "HGB"), ("XGBoost + MLP + TCN", "HGB + MLP + TCN"),
                     ("XGBoost + MLP + TCN", "MLP + TCN, logistic")):
        if new in preds and old in preds:
            lo, hi = class_bootstrap_diff(cls_cmp, y["compare"], preds[new], preds[old])
            rows.append({"draw": d, "model": f"{new} minus {old}",
                         "AP": average_precision_score(y["compare"], preds[new])
                               - average_precision_score(y["compare"], preds[old]),
                         "chosen": f"95% interval {lo:+.4f} to {hi:+.4f}"})
    print(f"draw {d} done ({time.time() - t0:.0f} s)")

res = pd.DataFrame(rows)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(res.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## Summary over the 5 draws

In [ ]:
fits = res[~res["model"].str.contains("minus")]
summary = fits.groupby("model", sort=False).agg(mean_AP=("AP", "mean"), min_AP=("AP", "min"), max_AP=("AP", "max"),
                                                 mean_chance_AP=("chance AP", "mean"))
summary["AP / chance"] = summary["mean_AP"] / summary["mean_chance_AP"]
print(summary.to_string(float_format=lambda v: f"{v:.4f}"))

diffs = res[res["model"].str.contains("minus")]
for name, g in diffs.groupby("model", sort=False):
    wins = int((g["AP"] > 0).sum())
    clear = int(g["chosen"].map(lambda s: float(s.split()[2]) > 0 or float(s.split()[4]) < 0).sum())
    print(f"\n{name}: mean difference {g['AP'].mean():+.4f}; first model higher in {wins} of {len(g)} draws; "
          f"interval excludes 0 in {clear} of {len(g)}")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
names = list(summary.index)
for i, n in enumerate(names):
    v = fits.loc[fits["model"] == n, "AP"].to_numpy()
    ax.scatter(np.full(len(v), i), v, color="tab:blue", alpha=0.6)
    ax.plot([i - 0.25, i + 0.25], [v.mean()] * 2, color="k", lw=2)
ax.axhline(fits["chance AP"].mean(), color="0.5", ls="--", label="chance (share of moments that are raises)")
ax.set_xticks(range(len(names)), names, rotation=15)
ax.set_ylabel("AP on the 20 comparison classes")
ax.set_title("Each dot is one draw of 20 / 20 / 20 train classes; black bar = mean")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Reading the result

* A difference counts only if it shows up in most draws **and** its 95% interval mostly stays on one side of 0.
  With 20 comparison classes and only a few dozen raises in them, differences of a few thousandths of AP are noise.
* HGB and XGBoost are close cousins (both histogram-based boosted trees with an L2 penalty on the leaves); the
  main extras XGBoost brings are up-weighting the rare raises and different pruning. Expect them to land close.
* If "XGBoost + MLP + TCN" does not beat "MLP + TCN, logistic", the trees are not adding to the two specialists,
  which is what the main notebook found with HGB.